# RGB-only Agentic Memory 实战教程

本 notebook 专门解释当前系统如何只依赖前端 RGB 视频建立 agentic memory。

目标链路：

```text
RGB video -> LingBot depth + c2w + intrinsics -> backprojection
         -> fixed-frame local submap -> overlap stability gate
         -> spatial memory -> VLM objects / Pi -> scene graph
         -> knowledge memory -> native agent reasoning
```

运行时不依赖 LiDAR、GT depth 或 GT pose。它们仅用于离线评测模型误差。

## 1. 先建立 notebook 环境

本 notebook 的所有示例都使用项目内的纯 Python 模块。LingBot/VLM 的外部服务不需要在本 notebook 中启动，目的是先把数据结构和记忆逻辑理解透。

In [1]:
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError(
        "This workshop requires Python 3.11+. Select the project .venv Python 3.12 kernel in VS Code, "
        "then run this cell again."
    )

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import numpy as np

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("project root:", project_root)
print("source path:", project_root / "src")

project root: /home/snt/projects/AgenticMemoryNav
source path: /home/snt/projects/AgenticMemoryNav/src


## 2. 为什么不能用点云质心判断稳定？

机器人向前移动时，看到的场景区域也会改变。因此点云质心变化并不意味着地图错误。

当前实现使用相邻局部点云的对称最近邻 overlap residual：

$$
r(P_t, P_{t+1}) = \frac{1}{2}(d(P_t,P_{t+1}) + d(P_{t+1},P_t))
$$

其中 $d(A,B)$ 是 $A$ 中每个点到 $B$ 最近点距离的平均值。窗口内最大残差小于阈值时，子地图可写入 spatial memory。

In [2]:
from agentic_memory_nav.common.types import MappingUpdate, Pose3D
from agentic_memory_nav.mapping.local_submap import LocalSubmapBuilder


def mapping_update(frame_index, offset_x):
    # 模拟 depth + c2w 已反投影得到的局部点云。
    cloud = np.array([
        [offset_x, 0.0, 1.0],
        [offset_x + 0.02, 0.0, 1.0],
        [offset_x, 0.02, 1.0],
    ], dtype=np.float32)
    return MappingUpdate(
        frame_id=f"frame_{frame_index:04d}",
        timestamp=float(frame_index),
        camera_pose=Pose3D(position=(offset_x, 0.0, 0.0)),
        depth=np.ones((2, 2), dtype=np.float32),
        confidence=np.ones((2, 2), dtype=np.float32),
        local_pointcloud=cloud,
        global_pointcloud=cloud,
        is_keyframe=True,
        map_version=frame_index + 1,
    )


builder = LocalSubmapBuilder(window_frames=3, frame_stride=1, stability_threshold_m=0.10)
submap = None
for index, offset_x in enumerate((0.00, 0.03, 0.06)):
    submap = builder.add(mapping_update(index, offset_x))

assert submap is not None
assert submap.stable
print("committed:", True)
print("stable:", submap.stable)
print("frames:", submap.frame_ids)
print("overlap residual (m):", round(submap.geometric_residual_m, 4))
print("points:", len(submap.points))

ImportError: cannot import name 'StrEnum' from 'enum' (/home/snt/.local/share/uv/python/cpython-3.10.18-linux-x86_64-gnu/lib/python3.10/enum.py)

上面的子地图是稳定的：机器人在移动，但重叠表面仍一致。

下面故意注入一个错误 frame。它可以对应 depth 崩坏、pose 跳变或不连续重定位。这个窗口不应该成为长期几何记忆。

In [ ]:
unstable_builder = LocalSubmapBuilder(
    window_frames=3,
    frame_stride=1,
    stability_threshold_m=0.10,
)
unstable_builder.add(mapping_update(0, 0.00))
unstable_builder.add(mapping_update(1, 0.03))
unstable = unstable_builder.add(mapping_update(2, 2.00))

assert unstable is not None
assert not unstable.stable
print("stable:", unstable.stable)
print("overlap residual (m):", round(unstable.geometric_residual_m, 4))
print("decision: do not commit spatial memory")

## 3. 从 VLM 的 2D bbox 到物体点云 $P_i$

VLM 负责语义：类别、属性、bbox、三元组候选。几何由 depth+c2w 决定。

当前最小可运行版本使用 bbox 生成 mask：

```text
VLM bbox -> binary mask -> selected depth pixels -> backprojection -> Pi artifact
```

未来可替换为 Grounding-DINO + SAM 或点云实例分割模型，后续 Scene Graph 和 Memory 的数据契约无需改变。

In [ ]:
from agentic_memory_nav.common.types import CameraIntrinsics, FrameObservation, ObjectObservation
from agentic_memory_nav.geometry.pointcloud_store import PointCloudStore
from agentic_memory_nav.perception.instance_segmentation import (
    BoundingBoxSegmenter,
    InstanceGeometryEnricher,
)

rgb = np.zeros((48, 64, 3), dtype=np.uint8)
depth = np.full((48, 64), 2.0, dtype=np.float32)
frame = FrameObservation(
    frame_id="pi_frame",
    timestamp=0.0,
    rgb=rgb,
    depth=depth,
    camera_intrinsics=CameraIntrinsics(60.0, 60.0, 32.0, 24.0, 64, 48),
    camera_pose=Pose3D(),
)
mapping = mapping_update(0, 0.0)
mapping.depth = depth
mapping.confidence = np.ones_like(depth)

observation = ObjectObservation(
    observation_id="obs_red_cube",
    category="cube",
    attributes={"color": "red"},
    bbox_2d=(20, 12, 44, 36),
    center_3d=(0.0, 0.0, 0.0),
    dimensions_3d=(0.0, 0.0, 0.0),
    confidence=0.9,
    timestamp=0.0,
    frame_id=frame.frame_id,
)

pi_store = PointCloudStore(Path("/tmp/agentic_memory_nav_notebook_pi"))
enricher = InstanceGeometryEnricher(BoundingBoxSegmenter(), pi_store)
enriched = enricher.enrich(frame, mapping, [observation])[0]

assert enriched.geometry is not None
print("Pi artifact:", enriched.geometry.artifact_path)
print("Pi point count:", enriched.geometry.point_count)
print("Pi centroid:", enriched.geometry.centroid_3d)
print("Pi dimensions:", enriched.geometry.dimensions_3d)

## 4. $P_i$ 如何进入 Scene Graph、Knowledge Memory、Reasoning 与 Navigation

点云不会直接塞进 SQLite。系统保存压缩 NPZ 工件引用、点数、中心、尺寸、置信度和 provenance。

- `SceneGraphUpdater` 把 room/object observation 合并为 node；
- 几何规则产生 `inside`、`near` 等 relation evidence；
- `KnowledgeMemory` 将 node 和三元组边投影为可检索事实；
- `NativeReasoner` 根据实体、属性和关系证据返回目标与证据；
- `RuleBasedPlanner` 使用 reasoner 的结果生成 `NAVIGATE`、`EXPLORE` 或需要验证的导航计划。

下一单元先建立 graph + memory graph，然后后续单元会让 planner 在它上面产生真实导航 action。

In [ ]:
from agentic_memory_nav.memory.knowledge_memory import KnowledgeMemory
from agentic_memory_nav.memory.sqlite_store import SQLiteMemory
from agentic_memory_nav.reasoning.native_reasoner import NativeReasoner
from agentic_memory_nav.scene_graph.graph import SceneGraph
from agentic_memory_nav.scene_graph.updater import SceneGraphUpdater

room = ObjectObservation(
    observation_id='obs_kitchen', category='kitchen', attributes={'kind': 'room'},
    bbox_2d=(0, 0, 64, 48), center_3d=(0.0, 0.0, 2.0),
    dimensions_3d=(6.0, 3.0, 6.0), confidence=0.99,
    timestamp=0.0, frame_id='pi_frame',
)
graph = SceneGraph()
SceneGraphUpdater(graph).update([room, enriched])

memory_path = Path('/tmp/agentic_memory_nav_notebook_knowledge.sqlite3')
if memory_path.exists():
    memory_path.unlink()
memory = SQLiteMemory(memory_path)
knowledge = KnowledgeMemory(memory)
created = knowledge.materialize(graph)
result = NativeReasoner(knowledge).resolve(
    graph, {'object': 'cube', 'color': 'red', 'room': 'kitchen'}
)

print('graph nodes:', len(graph.nodes()))
print('graph edges:', [(edge.relation, round(edge.confidence, 2)) for edge in graph.edges()])
print('knowledge facts:', created)
print('reasoning target:', result.target_id)
print('evidence ids:', result.evidence_ids)
print('requires verification:', result.requires_verification)
assert result.target_id is not None
memory.close()

## 5. Memory Graph 推理如何变成 Navigation Action

现在不只检查 `NativeReasoner` 能否找到目标，还要看它如何影响 navigation。

处理步骤：

1. `RuleBasedTaskParser` 将自然语言任务转成结构化目标；
2. `RuleBasedPlanner` 调用 `NativeReasoner`；
3. reasoner 从 scene graph 和 knowledge memory 的事实中验证：目标是否存在、是否在目标 room 内；
4. 证据充分时，planner 输出 `NAVIGATE`，其 waypoint 位于目标附近；
5. 没有目标证据时，planner 输出 `EXPLORE`；
6. 找到目标但 room relation 缺失时，plan 标记 `replan_required=True`，运行时应重新观察或执行验证。

当前轻量 parser 能稳定处理 `Find cube in the kitchen` 这类目标词。颜色、材质等细粒度语义由 VLM 和 memory graph 的节点属性保存；真实系统可以替换 parser 为更强的结构化语言解析器。

这就是 graph 构成的 memory graph 如何参与 agent 推理和导航，而不仅仅是一张可视化的图。

In [ ]:
from agentic_memory_nav.planning.rule_based_fallback import RuleBasedPlanner
from agentic_memory_nav.planning.task_parser import RuleBasedTaskParser

# 重新打开上一单元持久化的 memory graph，模拟 agent 在后续时刻恢复记忆。
navigation_memory = SQLiteMemory(memory_path)
# 当前 MVP parser 可稳定解析这个目标词；颜色仍保存在 graph node 的属性中。
task = RuleBasedTaskParser().parse("Find cube in the kitchen")
planner = RuleBasedPlanner(approach_distance=0.6)
plan = planner.plan(
    task=task,
    robot_pose=Pose3D(position=(0.0, 0.0, 0.0)),
    graph=graph,
    memory=navigation_memory,
    replan_reason="new stable RGB-only submap",
)

print("parsed goal:", task.parsed_goal)
print("action type:", plan.action.action_type.value)
print("navigation target node:", plan.action.target)
print("waypoint:", plan.action.waypoint)
print("plan confidence:", round(plan.confidence, 3))
print("replan required:", plan.replan_required)
print("information gaps:", plan.information_gaps)
print("reason:", plan.action.reason)

assert plan.action.action_type.value == "navigate"
assert plan.action.target == result.target_id
assert plan.action.waypoint is not None
navigation_memory.close()

### 缺少 memory graph 证据时会发生什么？

如果 graph 和 knowledge memory 中没有任务目标，agent 不会编造一个位置。planner 会生成 `EXPLORE` action，移动到一个受安全约束的探索 waypoint，等待新的 RGB observation、LingBot local submap 或 VLM 语义证据。

In [ ]:
empty_graph = SceneGraph()
empty_memory_path = Path("/tmp/agentic_memory_nav_notebook_empty.sqlite3")
if empty_memory_path.exists():
    empty_memory_path.unlink()
empty_memory = SQLiteMemory(empty_memory_path)

explore_plan = RuleBasedPlanner().plan(
    task=task,
    robot_pose=Pose3D(position=(0.0, 0.0, 0.0)),
    graph=empty_graph,
    memory=empty_memory,
)

print("action type:", explore_plan.action.action_type.value)
print("exploration waypoint:", explore_plan.action.waypoint)
print("information gaps:", explore_plan.information_gaps)
print("reason:", explore_plan.action.reason)

assert explore_plan.action.action_type.value == "explore"
assert explore_plan.replan_required
empty_memory.close()

## 6. Runtime 与离线研究的边界

运行时只依赖 RGB：

```text
RGB -> LingBot depth/c2w -> stable local submap -> spatial memory
RGB -> VLM -> semantic objects/triples -> scene graph + knowledge memory
memory graph -> native reasoning -> NAVIGATE / EXPLORE / VERIFY
```

运行时决策原则：

- memory graph 中存在目标、属性和 room relation 证据：生成 `NAVIGATE`；
- 目标存在但关系不完整：保留 information gap，要求重新观察或 `VERIFY`；
- graph/memory 中不存在目标：生成安全的 `EXPLORE` waypoint，不编造位置。

GT depth、GT c2w 和 LiDAR 用于离线研究：测量深度尺度漂移、点云误差和局部子地图稳定性。它们不是 RGB-only agent 的运行时前提。

当前系统会把 stable RGB-only submap 写入 spatial memory，同时保存 confidence、overlap residual 和 provenance。面对低置信度或冲突证据，agent 应重新观察或选择 `VERIFY`，而不是把所有几何关系当作绝对真相。